In [9]:
import glob
import os
import pandas as pd

# indiv_dir = '../data/indiv'
indiv_dir = './tmp_data'

In [10]:
# Get all the files in the directory
dirs = [d for d in sorted(glob.glob(indiv_dir + '/*')) if os.path.isdir(d)]

In [11]:
def stat_speaker(dir_speaker):
    d = dir_speaker
    speaker_id = d.split('/')[-1]
    segments_file = os.path.join(d, 'segments')
    text_file = os.path.join(d, 'text')

    df = pd.read_csv(segments_file, sep=' ', header=None, names=['utt_id', 'start', 'end'])
    total_num_utterances = df.shape[0]

    df['duration'] = df['end'] - df['start']
    total_duration_in_sec = df['duration'].sum()

    average_utterance_duration = total_duration_in_sec / total_num_utterances

    count_mora = 0
    count_bunsetsu = 0
    count_dis = 0
    count_fil = 0
    count_mask = 0
    count_sp = 0
    with open(text_file, 'r') as f:
        for lineno, line in enumerate(f):
            try:
                utt_id, text = line.strip().split(' ', 1)
            except:
                # エラーメッセージの表示
                print(f'Error in {text_file} at line {lineno}')
                raise
            moras = text.split(' ')
            for m in moras:
                if m == '|':
                    count_bunsetsu += 1
                else:
                    count_mora += 1
                if '+D' in m:
                    count_dis += 1
                if '+F' in m:
                    count_fil += 1
                if m == '<mask>':
                    count_mask += 1
                if m == '<sp>':
                    count_sp += 1
    return {
        'speaker_id': speaker_id,
        'total_num_utterances': total_num_utterances,
        'total_duration_in_sec': total_duration_in_sec,
        'average_utterance_duration': average_utterance_duration,
        'count_mora': count_mora,
        'mora_per_utterance': count_mora / total_num_utterances,
        'count_bunsetsu': count_bunsetsu,
        'bunsetsu_per_utterance': count_bunsetsu / total_num_utterances,
        'count_disfluency': count_dis,
        'count_filler': count_fil,
        'count_mask': count_mask,
        'count_sp': count_sp,
    }      

In [12]:
stats = []
for d in dirs:
    stats.append(stat_speaker(d))

In [13]:
df_stat = pd.DataFrame(stats, index=[s['speaker_id'] for s in stats])

In [14]:
df_stat.to_csv('stats_indiv_data.csv')